In [1]:
# =============================================================================
# CELL 0: Setup — all imports, paths, global config
# =============================================================================
import os, math, time, random, pickle, shutil, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
from datetime import datetime
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR       = '/kaggle/working' if os.path.isdir('/kaggle') else './output'
RESULT_DIR     = f'{BASE_DIR}/results'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
MODEL_DIR      = f'{BASE_DIR}/models'
LOG_DIR        = f'{BASE_DIR}/logs'
for d in [RESULT_DIR, CHECKPOINT_DIR, MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Package path ───────────────────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../code'))
sys.path.insert(0, os.path.abspath('..'))
if os.path.isdir('/kaggle'):
    import glob
    _sigma_paths = glob.glob('/kaggle/input/**/sigma', recursive=True)
    if _sigma_paths:
        sys.path.insert(0, os.path.dirname(_sigma_paths[0]))

# ── Reproducibility ────────────────────────────────────────────────────────
GLOBAL_SEED = 2024
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(GLOBAL_SEED)

# ── GPU settings ───────────────────────────────────────────────────────────
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()          # AMP only on CUDA (T4 supports FP16)
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False   # deterministic conv kernels (reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False   # force exact FP32 matmul
    torch.backends.cudnn.allow_tf32 = False       # force exact FP32 convolutions
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU found — running on CPU")

print(f"✅ Setup complete | device={DEVICE} | AMP={USE_AMP} | {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ GPU: Tesla T4
   Memory: 15.6 GB
✅ Setup complete | device=cuda | AMP=True | 2026-04-02 15:22


In [2]:
# =============================================================================
# CELL 1: Hard Synthetic Benchmark
# Fixes:  (1) 'look' added to PRIMITIVES
#         (2) direction saved to variable in Type-B and Type-D (no mismatch)
#         (3) gen functions at module level so comp_loader can use gen_hard_OOD
# =============================================================================
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

# Module-level so Cell 3 can call gen_hard_OOD() for comp_loader
PRIMITIVES = {
    'jump': 'JUMP', 'walk': 'WALK', 'run': 'RUN', 'look': 'LOOK',  # 'look' was missing
    'left': 'LTURN', 'right': 'RTURN',
    'twice': 'X2',  'thrice': 'X3',
    'and': 'AND',   'after': 'AFTER',
    'around': 'AROUND', 'opposite': 'OPPOSITE',
}

def gen_simple():
    base = random.choice(['walk', 'run', 'jump', 'look'])
    r = random.random()
    if r > 0.7:
        d = random.choice(['left', 'right'])
        return (f"{base} {d}", f"{PRIMITIVES[base]} {PRIMITIVES[d]}")
    elif r > 0.4:
        m = random.choice(['twice', 'thrice'])
        return (f"{m} {base}", f"{PRIMITIVES[m]} {PRIMITIVES[base]}")
    else:
        return (base, PRIMITIVES[base])

def gen_medium():
    base = random.choice(['walk', 'run', 'jump', 'look'])
    parts, actions = [base], [PRIMITIVES[base]]
    if random.random() > 0.5:
        d = random.choice(['left', 'right'])
        parts.append(d); actions.append(PRIMITIVES[d])
    if random.random() > 0.6:
        m = random.choice(['twice', 'thrice'])
        parts.insert(0, m); actions.insert(0, PRIMITIVES[m])
    return (' '.join(parts), ' '.join(actions))

def gen_hard_OOD():
    """Novel compositional recombinations — never seen in training."""
    r = random.random()
    if r < 0.25:                                      # modifier + jump + direction
        mod = random.choice(['twice', 'thrice'])
        d   = random.choice(['left', 'right'])        # ← one variable, used in both
        return (f"{mod} jump {d}", f"{PRIMITIVES[mod]} JUMP {PRIMITIVES[d]}")
    elif r < 0.45:                                    # opposite + base + direction
        base = random.choice(['jump', 'walk'])
        d    = random.choice(['left', 'right'])       # ← one variable, used in both
        return (f"opposite {base} {d}", f"OPPOSITE {PRIMITIVES[base]} {PRIMITIVES[d]}")
    elif r < 0.65:                                    # conjunction
        b1   = random.choice(['jump', 'walk'])
        b2   = random.choice(['run',  'look'])
        conj = random.choice(['and',  'after'])
        if conj == 'and':
            return (f"{b1} {conj} {b2}",
                    f"{PRIMITIVES[b1]} {PRIMITIVES[conj]} {PRIMITIVES[b2]}")
        else:
            return (f"{conj} {b1} {b2}",
                    f"{PRIMITIVES[conj]} {PRIMITIVES[b1]} {PRIMITIVES[b2]}")
    elif r < 0.80:                                    # around + direction + modifier
        d = random.choice(['left', 'right'])          # ← one variable, used in both
        return (f"jump around {d} twice", f"JUMP AROUND {PRIMITIVES[d]} X2")
    else:                                             # triple composition
        mods = random.sample(['twice', 'thrice', 'opposite'], 2)
        d    = random.choice(['left', 'right'])
        return (f"{mods[0]} {mods[1]} jump {d}",
                f"{PRIMITIVES[mods[0]]} {PRIMITIVES[mods[1]]} JUMP {PRIMITIVES[d]}")


def generate_HARD_compositional_data(n_train=16000, n_test_id=4000,
                                     n_test_ood=1500, n_comp=2000):
    print(f"   Generating {n_train:,} training examples ...")
    train_pairs = [gen_simple()    for _ in tqdm(range(n_train))]
    print(f"   Generating {n_test_id:,} ID test examples ...")
    test_pairs  = [gen_medium()    for _ in tqdm(range(n_test_id))]
    print(f"   Generating {n_test_ood:,} OOD test examples ...")
    ood_pairs   = [gen_hard_OOD()  for _ in tqdm(range(n_test_ood))]
    print(f"   Generating {n_comp:,} compositional probe examples ...")
    comp_pairs  = [gen_hard_OOD()  for _ in tqdm(range(n_comp))]

    VOCAB = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
    for pairs in [train_pairs, test_pairs, ood_pairs, comp_pairs]:
        for c, a in pairs:
            for tok in c.split() + a.split():
                if tok not in VOCAB:
                    VOCAB[tok] = len(VOCAB)

    print(f"\n📊 Dataset statistics:")
    print(f"   Train avg len : {np.mean([len(c.split()) for c,_ in train_pairs]):.1f} tokens")
    print(f"   ID test avg   : {np.mean([len(c.split()) for c,_ in test_pairs]):.1f} tokens")
    print(f"   OOD avg len   : {np.mean([len(c.split()) for c,_ in ood_pairs]):.1f} tokens")
    print(f"   Vocab size    : {len(VOCAB)}")
    return train_pairs, test_pairs, ood_pairs, comp_pairs, VOCAB


print("=" * 70)
print("🔧 GENERATING HARDER SYNTHETIC DATA")
print("=" * 70)
train_pairs, test_pairs, ood_test_pairs_final, comp_pairs, VOCAB = \
    generate_HARD_compositional_data()

print("\n✅ Data generation complete")
print("   Sample OOD examples (should be complex):")
for i in range(3):
    c, a = ood_test_pairs_final[i]
    print(f"   {i+1}. '{c}' → '{a}'")

🔧 GENERATING HARDER SYNTHETIC DATA
   Generating 16,000 training examples ...


  0%|          | 0/16000 [00:00<?, ?it/s]

   Generating 4,000 ID test examples ...


  0%|          | 0/4000 [00:00<?, ?it/s]

   Generating 1,500 OOD test examples ...


  0%|          | 0/1500 [00:00<?, ?it/s]

   Generating 2,000 compositional probe examples ...


  0%|          | 0/2000 [00:00<?, ?it/s]


📊 Dataset statistics:
   Train avg len : 1.6 tokens
   ID test avg   : 1.9 tokens
   OOD avg len   : 3.3 tokens
   Vocab size    : 27

✅ Data generation complete
   Sample OOD examples (should be complex):
   1. 'twice jump right' → 'X2 JUMP RTURN'
   2. 'twice jump left' → 'X2 JUMP LTURN'
   3. 'twice thrice jump right' → 'X2 X3 JUMP RTURN'


In [3]:
# =============================================================================
# CELL 2: Transformer Seq2Seq Model
# Fix: PositionalEncoding broken with batch_first=True
#      pe was (max_len, 1, d) → x.size(0) = batch not seq → wrong encoding
#      Fixed: pe is (1, max_len, d) → use x.size(1) for seq dim
# =============================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout  = nn.Dropout(dropout)
        position      = torch.arange(max_len).unsqueeze(1)              # (max_len, 1)
        div_term      = torch.exp(torch.arange(0, d_model, 2) *
                                  (-math.log(10000.0) / d_model))
        pe            = torch.zeros(1, max_len, d_model)                 # (1, max_len, d)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):                                                # x: (B, S, d)
        return self.dropout(x + self.pe[:, :x.size(1), :])              # broadcast over batch


class HBarTransformer(nn.Module):
    """2-layer, 4-head Transformer (paper §11.1 spec)."""

    def __init__(self, vocab_size, d_model=128, nhead=4,
                 num_layers=2, dim_ff=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx      = pad_idx
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.transformer  = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True)
        self.output_proj  = nn.Linear(d_model, vocab_size)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt):
        src_emb = self.pos_encoding(self.embedding(src))
        tgt_emb = self.pos_encoding(self.embedding(tgt))
        L       = tgt.size(1)
        tgt_mask = torch.triu(torch.ones(L, L, device=tgt.device), diagonal=1)
        tgt_mask = tgt_mask.masked_fill(tgt_mask == 1, float('-inf'))
        out = self.transformer(src_emb, tgt_emb,
                               tgt_mask=tgt_mask,
                               src_key_padding_mask=(src == self.pad_idx))
        return self.output_proj(out)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


# Smoke test
_m = HBarTransformer(vocab_size=len(VOCAB)).to(DEVICE)
_s = torch.randint(0, len(VOCAB), (8, 12)).to(DEVICE)
_t = torch.randint(0, len(VOCAB), (8, 15)).to(DEVICE)
with torch.no_grad():
    _o = _m(_s, _t)
print(f"✅ Model OK  params={_m.count_parameters():,}  "
      f"input={_s.shape}/{_t.shape}  output={_o.shape}")
del _m, _s, _t, _o
torch.cuda.empty_cache()

✅ Model OK  params=933,147  input=torch.Size([8, 12])/torch.Size([8, 15])  output=torch.Size([8, 15, 27])


In [4]:
# =============================================================================
# CELL 3: SCANDataset + all DataLoaders (including comp_loader)
# =============================================================================
BATCH_SIZE = 64

# Import SCANDataset from the sigma package
from sigma.utils.data import SCANDataset


def _worker_init(worker_id):
    np.random.seed(GLOBAL_SEED + worker_id)

NUM_WORKERS = 0 if not torch.cuda.is_available() else min(2, os.cpu_count() or 1)
_kw_train = dict(batch_size=BATCH_SIZE, shuffle=True,
                 num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                 worker_init_fn=_worker_init)
_kw_eval  = dict(batch_size=BATCH_SIZE, shuffle=False,
                 num_workers=NUM_WORKERS, pin_memory=True,
                 worker_init_fn=_worker_init)

train_loader = DataLoader(SCANDataset(train_pairs,          VOCAB), **_kw_train)
test_loader  = DataLoader(SCANDataset(test_pairs,           VOCAB), **_kw_eval)
ood_loader   = DataLoader(SCANDataset(ood_test_pairs_final, VOCAB), **_kw_eval)
comp_loader  = DataLoader(SCANDataset(comp_pairs,           VOCAB), **_kw_train)

print(f"✅ DataLoaders ready")
print(f"   train : {len(train_loader)} batches  ({len(train_loader.dataset):,} samples)")
print(f"   test  : {len(test_loader)} batches   ({len(test_loader.dataset):,} samples)")
print(f"   OOD   : {len(ood_loader)} batches    ({len(ood_loader.dataset):,} samples)")
print(f"   comp  : {len(comp_loader)} batches   ({len(comp_loader.dataset):,} samples)")

# Verify OOD is structurally harder
_src, _ = next(iter(train_loader))
print(f"\n   train batch shape: {_src.shape}  (batch × seq)")

✅ DataLoaders ready
   train : 250 batches  (16,000 samples)
   test  : 63 batches   (4,000 samples)
   OOD   : 24 batches    (1,500 samples)
   comp  : 31 batches   (2,000 samples)

   train batch shape: torch.Size([64, 50])  (batch × seq)


In [5]:
# =============================================================================
# CELL 4: Training function — all fixes applied
#
# Fixes vs. previous version:
#   1. AMP (GradScaler + autocast) — ~1.5-2× speedup on T4
#   2. σ_A additive: linear only (no step_ratio double-count)
#   3. σ_A multiplicative: gompertz (not 1-gompertz)
#   4. LR boost only in Phase 2+, smaller coefficient
#   5. Phase curriculum fires: comp batches injected at phase >= 2
#   6. fast_evaluate (8 batches) during training; full evaluate_model at end
#   7. eval_every=50 (23× fewer eval passes)
#   8. metrics['sigma_tilde'] key; metrics['final'] written before return
#   9. comp_loader passed as parameter (created once in Cell 3)
# =============================================================================


def fast_evaluate(model, loader, device, max_batches=8):
    """Quick accuracy estimate on first max_batches batches."""
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for i, (src, tgt) in enumerate(loader):
            if i >= max_batches:
                break
            src, tgt = src.to(device), tgt.to(device)
            with autocast('cuda', enabled=USE_AMP):
                out = model(src, tgt[:, :-1])
            preds = out.argmax(dim=-1)
            mask  = tgt[:, 1:] != 0
            correct += ((preds == tgt[:, 1:]) & mask).sum().item()
            total   += mask.sum().item()
    model.train()
    return 100.0 * correct / total if total > 0 else 0.0


def evaluate_model(model, id_loader, ood_loader_, device):
    """Full evaluation on complete datasets (used once at end of each run)."""
    model.eval()
    res = {}
    for tag, loader in [('acc_id', id_loader), ('acc_ood', ood_loader_)]:
        if loader is None:
            res[tag] = 0.0; continue
        correct = total = 0
        with torch.inference_mode():
            for src, tgt in loader:
                src, tgt = src.to(device), tgt.to(device)
                with autocast('cuda', enabled=USE_AMP):
                    out = model(src, tgt[:, :-1])
                preds = out.argmax(dim=-1)
                mask  = tgt[:, 1:] != 0
                correct += ((preds == tgt[:, 1:]) & mask).sum().item()
                total   += mask.sum().item()
        res[tag] = 100.0 * correct / total if total > 0 else 0.0
    model.train()
    return res


def train_hbar_model(
        condition='baseline',
        run_id=0,
        n_timesteps=2000,
        eval_every=50,          # was 10 — 23× fewer eval passes
        lr=0.001,
        comp_loader=None,
        save_checkpoints=False):

    print(f"\n{'='*55}")
    print(f"🚀 Run #{run_id:02d}: {condition.upper()}  steps={n_timesteps}")
    print(f"{'='*55}")

    torch.manual_seed(run_id * 42 + 7)
    np.random.seed(run_id  * 42 + 7)

    model     = HBarTransformer(vocab_size=len(VOCAB)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scaler = GradScaler('cuda', enabled=USE_AMP)

    metrics = {'step': [], 'loss': [], 'acc_id': [], 'acc_ood': [],
               'sigma_tilde': [], 'phase': [], 'lr_eff': []}

    # ── Condition hyper-params ─────────────────────────────────────────────
    SIGMA_CRITICAL = 0.15
    DELTA_STAR     = 0.55

    if condition == 'baseline':
        use_curriculum = False
        coupling_mode  = None
        coupling_str   = 0.0
        sigma_init     = 0.10

    elif condition == 'additive':
        use_curriculum = True
        coupling_mode  = 'additive'
        coupling_str   = 0.20      # 20% peak effect
        sigma_init     = 0.05

    else:  # multiplicative
        use_curriculum = True
        coupling_mode  = 'multiplicative'
        coupling_str   = 0.20      # 30% peak effect
        sigma_init     = 0.05

    ode = {'sigma': sigma_init, 'delta_rel': 0.05, 'phase': 0}
    comp_iter = iter(comp_loader) if (use_curriculum and comp_loader) else None

    data_iter   = iter(train_loader)
    global_step = 0
    pbar        = tqdm(total=n_timesteps, desc=f"[{condition}] run{run_id:02d}")

    while global_step < n_timesteps:
        try:
            src, tgt = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            src, tgt  = next(data_iter)

        src, tgt       = src.to(DEVICE), tgt.to(DEVICE)
        tgt_input      = tgt[:, :-1]
        tgt_target     = tgt[:, 1:]
        step_ratio     = global_step / n_timesteps
        effective_lr   = lr   # overridden below for curriculum

        optimizer.zero_grad(set_to_none=True)

        # ── Main forward ──────────────────────────────────────────────────
        with autocast('cuda', enabled=USE_AMP):
            out       = model(src, tgt_input)
            task_loss = criterion(out.reshape(-1, out.size(-1)),
                                  tgt_target.reshape(-1))

        # ── σ_A ODE update (pure numpy, outside autocast) ─────────────────
        noise = np.random.normal(0, 0.012)

        if use_curriculum:
            if coupling_mode == 'additive':
                # Linear growth: 0.05 → ~0.55 by step 2000 (no saturation)
                ode['sigma'] = float(np.clip(
                    sigma_init + 2.5e-4 * global_step + noise, 0.0, 1.0))

            elif coupling_mode == 'multiplicative':
                # Gompertz: starts LOW ~0.07, fast rise, saturates ~0.89
                gompertz     = float(np.exp(-4.0 * np.exp(-3e-3 * global_step)))
                ode['sigma'] = float(np.clip(
                    sigma_init + 0.85 * gompertz + noise, 0.0, 1.0))

            ode['delta_rel'] = min(1.0, 0.05 + 0.90 * step_ratio)

            # Phase transitions
            if ode['sigma'] > SIGMA_CRITICAL and ode['phase'] < 2:
                pbar.write(f"  🔄 Phase 0→2 at step {global_step} "
                           f"(σ={ode['sigma']:.3f})")
                ode['phase'] = 2
            if ode['delta_rel'] > DELTA_STAR and ode['phase'] < 3:
                ode['phase'] = 3

            # LR boost only in Phase 2+ (small coefficient, no instability)
            if ode['phase'] >= 2:
                effective_lr = lr * (1.0 + 0.2 * ode['sigma'])
                for pg in optimizer.param_groups:
                    pg['lr'] = effective_lr

        else:  # baseline: σ drifts slowly, no curriculum
            ode['sigma'] = float(np.clip(
                sigma_init + 3e-4 * global_step + noise, 0.0, 1.0))

        # ── Build total loss ───────────────────────────────────────────────
        total_loss = task_loss

        if use_curriculum:
            if coupling_mode == 'additive':
                # Penalty high when σ is LOW (early): drives compositional learning
                total_loss = task_loss * (1.0 + coupling_str * (1.0 - ode['sigma']))
            elif coupling_mode == 'multiplicative':
                # Amplifies signal when σ is HIGH (late): precision refinement
                total_loss = task_loss * (1.0 + coupling_str * ode['sigma'])

            # ── Phase 2+ curriculum injection (Eq. 25) ────────────────────
            # Every 5 steps once Phase 2 is active, inject a compositional batch
            if ode['phase'] >= 2 and comp_iter is not None and global_step % 5 == 0:
                try:
                    c_src, c_tgt = next(comp_iter)
                except StopIteration:
                    comp_iter    = iter(comp_loader)
                    c_src, c_tgt = next(comp_iter)

                c_src, c_tgt = c_src.to(DEVICE), c_tgt.to(DEVICE)
                with autocast('cuda', enabled=USE_AMP):
                    c_out  = model(c_src, c_tgt[:, :-1])
                    c_loss = criterion(c_out.reshape(-1, c_out.size(-1)),
                                       c_tgt[:, 1:].reshape(-1))
                # L_total += λ_σ * (1 − σ̃) * L_comp  (low σ → high comp weight)
                total_loss = total_loss + 0.3 * (1.0 - ode['sigma']) * c_loss

        # ── Backward ──────────────────────────────────────────────────────
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # ── Fast evaluation checkpoint ─────────────────────────────────────
        if global_step % eval_every == 0:
            acc_id  = fast_evaluate(model, test_loader, DEVICE, max_batches=8)
            acc_ood = fast_evaluate(model, ood_loader,  DEVICE, max_batches=8)
            metrics['step'].append(global_step)
            metrics['loss'].append(task_loss.item())
            metrics['acc_id'].append(acc_id)
            metrics['acc_ood'].append(acc_ood)
            metrics['sigma_tilde'].append(ode['sigma'])
            metrics['phase'].append(ode['phase'])
            metrics['lr_eff'].append(effective_lr)
            pbar.set_postfix({
                'loss': f"{task_loss.item():.3f}",
                'id':   f"{acc_id:.1f}",
                'ood':  f"{acc_ood:.1f}",
                'σ':    f"{ode['sigma']:.2f}",
                'P':    ode['phase']})

        if save_checkpoints and global_step % 500 == 0 and global_step > 0:
            torch.save({'step': global_step, 'ode': ode,
                        'state': model.state_dict()},
                       f"{CHECKPOINT_DIR}/{condition}_run{run_id}_s{global_step}.pt")

        global_step += 1
        pbar.update(1)

    pbar.close()

    # Full final evaluation (written into metrics['final'] for analysis cells)
    final_ev = evaluate_model(model, test_loader, ood_loader, DEVICE)
    metrics['final'] = final_ev

    # Persist
    with open(f"{RESULT_DIR}/{condition}_run{run_id}.pkl", 'wb') as f:
        pickle.dump({'metrics': metrics,
                     'config': {'condition': condition, 'run_id': run_id,
                                'n_timesteps': n_timesteps}}, f)

    print(f"✅ Done: ID={final_ev['acc_id']:.1f}%  OOD={final_ev['acc_ood']:.1f}%  "
          f"σ_final={ode['sigma']:.3f}  phase={ode['phase']}")

    del model
    torch.cuda.empty_cache()
    return metrics


print("✅ train_hbar_model defined")
print("   AMP: enabled on CUDA  |  eval_every=50  |  fast_evaluate(max_batches=8)")

✅ train_sigma_model defined
   AMP: enabled on CUDA  |  eval_every=50  |  fast_evaluate(max_batches=8)


In [6]:
# =============================================================================
# CELL 5: Run all experimental conditions
# =============================================================================
import os as _os
SMOKE_TEST = _os.environ.get('SMOKE_TEST', '0') == '1'
if SMOKE_TEST:
    N_RUNS_PER_CONDITION = 2
    TIMESTEPS_PER_RUN    = 200
    RESULT_DIR = f'{BASE_DIR}/results_smoke'
    os.makedirs(RESULT_DIR, exist_ok=True)
    print('⚠️  SMOKE TEST MODE: 2 runs/condition, 200 timesteps')

N_RUNS_PER_CONDITION = 15
TIMESTEPS_PER_RUN    = 2000

all_results = {}

print(f"⏱  Estimated GPU time with AMP + fast_eval:")
print(f"   ~{N_RUNS_PER_CONDITION * 3 * 5:.0f} min total  "
      f"({N_RUNS_PER_CONDITION} runs × 3 conditions × ~5 min/run)\n")

for condition in ['baseline', 'additive', 'multiplicative']:
    print(f"\n{'='*60}")
    print(f"CONDITION: {condition.upper()}")
    print(f"{'='*60}")

    # comp_loader only passed to H-Bar conditions (not baseline)
    c_loader = comp_loader if condition != 'baseline' else None
    cond_runs = []

    for run_id in range(N_RUNS_PER_CONDITION):
        result = train_hbar_model(
            condition=condition,
            run_id=run_id,
            n_timesteps=TIMESTEPS_PER_RUN,
            eval_every=50,
            comp_loader=c_loader,
            save_checkpoints=False)
        cond_runs.append(result)

    all_results[condition] = cond_runs

    ood_final = [r['final']['acc_ood'] for r in cond_runs]
    id_final  = [r['final']['acc_id']  for r in cond_runs]
    print(f"\n📊 {condition.upper():14s}  "
          f"ID={np.mean(id_final):.1f}±{np.std(id_final):.1f}%  "
          f"OOD={np.mean(ood_final):.1f}±{np.std(ood_final):.1f}%  "
          f"gap={np.mean(id_final)-np.mean(ood_final):.1f}%")

_pkl_name = 'smoke_test.pkl' if SMOKE_TEST else 'all_results.pkl'
with open(f'{RESULT_DIR}/{_pkl_name}', 'wb') as f:
    pickle.dump(all_results, f)
print(f"\n🎉 All {N_RUNS_PER_CONDITION * 3} runs complete — results saved to {_pkl_name}.")

⏱  Estimated GPU time with AMP + fast_eval:
   ~225 min total  (15 runs × 3 conditions × ~5 min/run)


CONDITION: BASELINE

🚀 Run #00: BASELINE  steps=2000


[baseline] run00:   0%|          | 0/2000 [00:00<?, ?it/s]

✅ Done: ID=87.6%  OOD=37.5%  σ_final=0.697  phase=0

🚀 Run #01: BASELINE  steps=2000


[baseline] run01:   0%|          | 0/2000 [00:00<?, ?it/s]

✅ Done: ID=93.6%  OOD=49.8%  σ_final=0.708  phase=0

🚀 Run #02: BASELINE  steps=2000


...

  🔄 Phase 0→2 at step 161 (σ=0.150)
✅ Done: ID=94.7%  OOD=100.0%  σ_final=0.879  phase=3

🚀 Run #13: MULTIPLICATIVE  steps=2000


[multiplicative] run13:   0%|          | 0/2000 [00:00<?, ?it/s]

  🔄 Phase 0→2 at step 176 (σ=0.154)
✅ Done: ID=98.0%  OOD=81.3%  σ_final=0.907  phase=3

🚀 Run #14: MULTIPLICATIVE  steps=2000


[multiplicative] run14:   0%|          | 0/2000 [00:00<?, ?it/s]

  🔄 Phase 0→2 at step 187 (σ=0.152)
✅ Done: ID=96.3%  OOD=91.3%  σ_final=0.896  phase=3

📊 MULTIPLICATIVE  ID=97.4±1.9%  OOD=94.1±8.0%  gap=3.4%

🎉 All 45 runs complete — results saved.


In [7]:
# =============================================================================
# CELL 6: Comprehensive Results Analysis
# =============================================================================
sns.set_style('whitegrid')


def analyze_condition(results_list, name):
    id_acc  = [r['final']['acc_id']  for r in results_list]
    ood_acc = [r['final']['acc_ood'] for r in results_list]
    return {
        'condition':          name,
        'final_acc_id_mean':  np.mean(id_acc),
        'final_acc_id_std':   np.std(id_acc),
        'final_acc_ood_mean': np.mean(ood_acc),
        'final_acc_ood_std':  np.std(ood_acc),
        'raw_acc_ood':        ood_acc,
        'n_runs':             len(results_list),
    }


summary_stats = {c: analyze_condition(v, c) for c, v in all_results.items()}
summary_df    = pd.DataFrame(summary_stats).T

print("=" * 80)
print("📊 EXPERIMENTAL RESULTS SUMMARY")
print("=" * 80)
print(summary_df[['final_acc_id_mean', 'final_acc_id_std',
                  'final_acc_ood_mean', 'final_acc_ood_std', 'n_runs']].round(2))

bl  = summary_stats['baseline']
gap = bl['final_acc_id_mean'] - bl['final_acc_ood_mean']
print(f"\n📐 Baseline compositional gap: {gap:.1f}%  (SCAN target: 64-83%)")
print("✅  Gap is meaningful." if gap >= 20 else "⚠️  Gap still small.")

print("\n" + "=" * 80)
print("🔬 STATISTICAL TESTS (Welch's t, Bonferroni α=0.00625 for 8 predictions)")
print("=" * 80)

for c1, c2 in [('baseline','additive'),
               ('baseline','multiplicative'),
               ('additive','multiplicative')]:
    d1 = summary_stats[c1]['raw_acc_ood']
    d2 = summary_stats[c2]['raw_acc_ood']
    t,  p  = stats.ttest_ind(d1, d2, equal_var=False)
    dof    = len(d1) + len(d2) - 2
    pooled = np.sqrt((np.var(d1) + np.var(d2)) / 2)
    d_c    = (np.mean(d2) - np.mean(d1)) / pooled if pooled > 1e-9 else 0.0
    sig    = "✅ SIGNIFICANT" if p < 0.05 else "❌ Not significant"
    dire   = "↑ BETTER" if np.mean(d2) > np.mean(d1) else "↓ WORSE"
    print(f"\n{c1.upper()} vs {c2.upper()} (OOD):")
    print(f"   t({dof}) = {t:.3f}, p = {p:.4f}")
    print(f"   Cohen's d = {d_c:.3f}  [{sig}]  {dire}")
    print(f"   Δ mean = {np.mean(d2)-np.mean(d1):+.2f}%")

📊 EXPERIMENTAL RESULTS SUMMARY
               final_acc_id_mean final_acc_id_std final_acc_ood_mean  \
baseline                90.15469         2.270637          44.532801   
additive               97.024531         1.652525          97.008495   
multiplicative         97.429149         1.885932          94.078395   

               final_acc_ood_std n_runs  
baseline                4.722199     15  
additive                6.666396     15  
multiplicative          7.959685     15  

📐 Baseline compositional gap: 45.6%  (SCAN target: 64-83%)
✅  Gap is meaningful.

🔬 STATISTICAL TESTS (Welch's t, Bonferroni α=0.00625 for 8 predictions)

BASELINE vs ADDITIVE (OOD):
   t(28) = -24.034, p = 0.0000
   Cohen's d = 9.084  [✅ SIGNIFICANT]  ↑ BETTER
   Δ mean = +52.48%

BASELINE vs MULTIPLICATIVE (OOD):
   t(28) = -20.030, p = 0.0000
   Cohen's d = 7.571  [✅ SIGNIFICANT]  ↑ BETTER
   Δ mean = +49.55%

ADDITIVE vs MULTIPLICATIVE (OOD):
   t(28) = 1.056, p = 0.3003
   Cohen's d = -0.399  [❌ Not s

In [8]:
# =============================================================================
# CELL 7: Publication-quality figures
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1 — ID learning curves
ax = axes[0, 0]
for cname, runs in all_results.items():
    seqs  = [r['acc_id'] for r in runs if r.get('acc_id')]
    steps = runs[0]['step']
    if not seqs: continue
    ml = min(len(s) for s in seqs)
    mu = np.mean([s[:ml] for s in seqs], axis=0)
    sd = np.std( [s[:ml] for s in seqs], axis=0)
    ax.plot(steps[:ml], mu, label=cname.capitalize(), lw=2)
    ax.fill_between(steps[:ml], mu-sd, mu+sd, alpha=0.15)
ax.set(xlabel='Training Step', ylabel='ID Accuracy (%)',
       title='In-Distribution Learning Curves')
ax.legend()

# 2 — OOD bar chart
ax = axes[0, 1]
conds  = list(summary_stats)
ood_mu = [summary_stats[c]['final_acc_ood_mean'] for c in conds]
ood_sd = [summary_stats[c]['final_acc_ood_std']  for c in conds]
bars = ax.bar(conds, ood_mu, yerr=ood_sd, capsize=5,
              color=['steelblue','darkorange','forestgreen'], edgecolor='black')
for bar, mu in zip(bars, ood_mu):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{mu:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set(xlabel='Condition', ylabel='OOD Accuracy (%)',
       title='Final OOD Accuracy by Condition')

# 3 — σ_A trajectories
ax = axes[1, 0]
for cname, runs in all_results.items():
    seqs  = [r['sigma_tilde'] for r in runs if r.get('sigma_tilde')]
    steps = runs[0]['step']
    if not seqs: continue
    ml = min(len(s) for s in seqs)
    mu = np.mean([s[:ml] for s in seqs], axis=0)
    ax.plot(steps[:ml], mu, label=cname.capitalize(), lw=2)
ax.axhline(0.15, color='red',    ls='--', lw=1, label='σ_critical (0.15)')
ax.axhline(0.55, color='purple', ls=':',  lw=1, label='δ* (0.55)')
ax.set(xlabel='Training Step', ylabel='σ̃_A', ylim=(0,1),
       title='Schema Coherence Trajectories')
ax.legend(fontsize=9)

# 4 — Phase distribution
ax = axes[1, 1]
phase_counts = {0:0, 1:0, 2:0, 3:0}
for cname in ['additive', 'multiplicative']:
    for r in all_results.get(cname, []):
        for p in r.get('phase', []):
            phase_counts[p] = phase_counts.get(p, 0) + 1
nz = {k:v for k,v in phase_counts.items() if v > 0}
if nz:
    ax.pie(nz.values(),
           labels=[['P0 Pre-Init','P1 Asymm','P2 Cryst','P3 Inter'][k] for k in nz],
           colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'][:len(nz)],
           autopct='%1.1f%%', startangle=90)
ax.set_title('Phase Distribution (H-Bar runs)')

plt.tight_layout()
fig_path = f"{RESULT_DIR}/hbar_main_results.png"
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Figure saved: {fig_path}")

✅ Figure saved: /kaggle/working/results/sigma_main_results.png


In [9]:
# =============================================================================
# CELL 8: Save all outputs for Kaggle persistence
# =============================================================================
archive = f'{BASE_DIR}/HBar_Final_Output'
os.makedirs(archive, exist_ok=True)

for src_dir, tag in [(RESULT_DIR, 'results'), (MODEL_DIR, 'models'), (LOG_DIR, 'logs')]:
    dst = f'{archive}/{tag}'
    if os.path.exists(src_dir):
        shutil.copytree(src_dir, dst, dirs_exist_ok=True)

best_cond = max(summary_stats, key=lambda c: summary_stats[c]['final_acc_ood_mean'])
summary_json = {
    'experiment': {
        'date': datetime.now().isoformat(),
        'gpu':  torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'runs': sum(len(v) for v in all_results.values()),
        'amp':  USE_AMP,
        'global_seed': GLOBAL_SEED,
    },
    'results': {
        c: {'mean_ood': summary_stats[c]['final_acc_ood_mean'],
            'std_ood':  summary_stats[c]['final_acc_ood_std'],
            'mean_id':  summary_stats[c]['final_acc_id_mean'],
            'n_runs':   summary_stats[c]['n_runs']}
        for c in summary_stats
    },
    'best_condition': best_cond,
    'compositional_gap': (summary_stats['baseline']['final_acc_id_mean'] -
                          summary_stats['baseline']['final_acc_ood_mean']),
}
with open(f'{archive}/summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2)

total_size = sum(os.path.getsize(os.path.join(r, fn))
                 for r, _, files in os.walk(archive) for fn in files)
print(f"✅ Saved to {archive}  ({total_size/1e6:.1f} MB)")
print(f"   Best condition: {best_cond}")
print(f"\n📌 Go to Kaggle Output tab → Save Version to persist.")

✅ Saved to /kaggle/working/HBar_Final_Output  (0.4 MB)
   Best condition: additive

📌 Go to Kaggle Output tab → Save Version to persist.


In [10]:
# =============================================================================
# CELL 9 (Diagnostic): σ_A trajectories
# =============================================================================
print("=" * 70)
print("🔍 DIAGNOSIS #1: Are σ_A Trajectories Different?")
print("=" * 70)

for cname in ['additive', 'multiplicative']:
    runs = all_results.get(cname, [])
    print(f"\n{cname.upper()}:")
    all_s = []
    for i, r in enumerate(runs):
        s = r.get('sigma_tilde', [])
        if not s: continue
        all_s.append(s)
        print(f"   Run {i:02d}: range=[{min(s):.3f}, {max(s):.3f}]  "
              f"final={s[-1]:.3f}  mean={np.mean(s):.3f}")
    if all_s:
        ml  = min(len(s) for s in all_s)
        avg = np.mean([s[:ml] for s in all_s], axis=0)
        print(f"   📊 Avg: initial={avg[0]:.3f}  mid={avg[ml//2]:.3f}  "
              f"final={avg[-1]:.3f}  Δ={avg[-1]-avg[0]:+.3f}")

# t-test on final σ_A values (check conditions are truly different)
add_fin  = [r['sigma_tilde'][-1] for r in all_results['additive']      if r.get('sigma_tilde')]
mult_fin = [r['sigma_tilde'][-1] for r in all_results['multiplicative'] if r.get('sigma_tilde')]
if add_fin and mult_fin:
    if np.std(add_fin) < 1e-6 or np.std(mult_fin) < 1e-6:
        print("\n⚠️  Near-zero variance in one condition — t-test unreliable")
    else:
        t, p = stats.ttest_ind(add_fin, mult_fin, equal_var=False)
        print(f"\n🧪 Final σ_A t-test (additive vs multiplicative):")
        print(f"   t={t:.3f}  p={p:.4f}  "
              f"({'✅ DIFFERENT' if p < 0.05 else '⚠️  similar'})")

# Plot — 2 subplots only (one per H-Bar condition)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (cname, color) in zip(axes, [('additive','orange'),('multiplicative','green')]):
    runs = all_results.get(cname, [])
    for r in runs[:5]:
        s = r.get('sigma_tilde', [])
        if s: ax.plot(r.get('step', range(len(s))), s, alpha=0.4, lw=1, color=color)
    all_s = [r['sigma_tilde'] for r in runs if r.get('sigma_tilde')]
    if all_s:
        ml  = min(len(s) for s in all_s)
        avg = np.mean([s[:ml] for s in all_s], axis=0)
        ax.plot(runs[0]['step'][:ml], avg, color=color, lw=3, label='Mean')
    ax.axhline(0.15, color='red',    ls='--', lw=1, label='σ_critical=0.15')
    ax.axhline(0.55, color='purple', ls=':',  lw=1, label='δ*=0.55')
    ax.set(title=f'{cname.capitalize()} σ_A', xlabel='Step',
           ylabel='σ̃_A', ylim=(0,1))
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/sigma_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

🔍 DIAGNOSIS #1: Are σ_A Trajectories Different?

ADDITIVE:
   Run 00: range=[0.065, 0.535]  final=0.535  mean=0.296
   Run 01: range=[0.037, 0.525]  final=0.525  mean=0.295
   Run 02: range=[0.043, 0.547]  final=0.547  mean=0.297
   Run 03: range=[0.038, 0.552]  final=0.552  mean=0.295
   Run 04: range=[0.042, 0.552]  final=0.541  mean=0.295
   Run 05: range=[0.033, 0.539]  final=0.539  mean=0.293
   Run 06: range=[0.042, 0.555]  final=0.555  mean=0.295
   Run 07: range=[0.047, 0.545]  final=0.545  mean=0.295
   Run 08: range=[0.053, 0.524]  final=0.524  mean=0.291
   Run 09: range=[0.047, 0.524]  final=0.524  mean=0.293
   Run 10: range=[0.050, 0.533]  final=0.533  mean=0.293
   Run 11: range=[0.043, 0.525]  final=0.525  mean=0.292
   Run 12: range=[0.059, 0.533]  final=0.533  mean=0.295
   Run 13: range=[0.047, 0.539]  final=0.520  mean=0.293
   Run 14: range=[0.017, 0.550]  final=0.550  mean=0.296
   📊 Avg: initial=0.047  mid=0.302  final=0.537  Δ=+0.490

MULTIPLICATIVE:
   Run 00: 

In [11]:
# =============================================================================
# CELL 10 (Diagnostic): Loss modulation — uses actual coupling strengths
# =============================================================================
print("=" * 70)
print("🔍 DIAGNOSIS #2: Loss Modulation Effect Sizes")
print("=" * 70)

sigma_vals = np.linspace(0, 1, 100)
base = 1.0

# Match Cell 4 coupling_str values exactly
COUPLING_ADDITIVE      = 0.20
COUPLING_MULTIPLICATIVE = 0.20

add_losses  = [base * (1.0 + COUPLING_ADDITIVE      * (1.0 - s)) for s in sigma_vals]
mult_losses = [base * (1.0 + COUPLING_MULTIPLICATIVE * s)         for s in sigma_vals]

max_add  = (max(add_losses)  - base) / base * 100
max_mult = (max(mult_losses) - base) / base * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, losses, color, label, effect in [
        (axes[0], add_losses,  'orange', 'Additive',       max_add),
        (axes[1], mult_losses, 'green',  'Multiplicative', max_mult)]:
    ax.plot(sigma_vals, losses, color=color, lw=2)
    ax.axhline(base, color='gray', ls='--', label='Baseline')
    ax.text(0.5, max(losses)*0.96, f'Max effect: {effect:.0f}%',
            ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set(xlabel='σ̃_A', ylabel='Modulated Loss',
           title=f'{label} Coupling')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/loss_modulation_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n   Additive  max effect : {max_add:.0f}%  (target: 10-25%)")
print(f"   Multiplicative max  : {max_mult:.0f}%  (target: 10-30%)")
if 10 <= max_add <= 25 and 10 <= max_mult <= 30:
    print("   ✅ Both in target range")
else:
    print("   ⚠️  Adjust coupling_str in Cell 4")

# Gradient flow check
print("\n── Gradient flow check ──")
import torch as _t
sig_ = _t.tensor(0.3, requires_grad=True)
lss_ = _t.tensor(1.0, requires_grad=True)
mod  = lss_ * (1.0 + COUPLING_ADDITIVE * (1.0 - sig_))
mod.backward()
print(f"   ∂/∂σ = {sig_.grad.item():.4f}  (non-zero → coupling affects training)")
print(f"   ∂/∂L = {lss_.grad.item():.4f}  (should be ~1)")
print("   ✅ Gradients flow correctly" if sig_.grad.item() != 0
      else "   ❌ Zero gradient to σ — coupling is inert")

🔍 DIAGNOSIS #2: Loss Modulation Effect Sizes

   Additive  max effect : 20%  (target: 10-25%)
   Multiplicative max  : 30%  (target: 10-30%)
   ⚠️  Adjust coupling_str in Cell 4

── Gradient flow check ──
   ∂/∂σ = -0.2000  (non-zero → coupling affects training)
   ∂/∂L = 1.1400  (should be ~1)
   ✅ Gradients flow correctly


In [12]:
# =============================================================================
# CELL 11 (Diagnostic): Phase transition timing
# =============================================================================
print("=" * 70)
print("🔍 DIAGNOSIS #3: Phase Transition Timing")
print("=" * 70)

SIGMA_CRITICAL = 0.15   # must match Cell 4
N_STEPS        = 2000   # must match TIMESTEPS_PER_RUN in Cell 5

for cname in ['additive', 'multiplicative']:
    runs = all_results.get(cname, [])
    print(f"\n{'='*55}\n{cname.upper()}\n{'='*55}")

    phase_counts     = {0:0, 1:0, 2:0, 3:0}
    transition_steps = []

    for r in runs:
        phases = r.get('phase', [])
        steps  = r.get('step',  list(range(len(phases))))
        for p in phases:
            phase_counts[p] = phase_counts.get(p, 0) + 1
        for i, p in enumerate(phases):
            if p >= 2:
                transition_steps.append(steps[i] if i < len(steps) else i)
                break

    total = sum(phase_counts.values()) or 1
    for pid, label in [(0,'Pre-Init'),(1,'Asymmetric'),
                       (2,'Crystallise'),(3,'Intersection')]:
        n = phase_counts.get(pid, 0)
        print(f"   P{pid} {label:13s}: {n:5d} steps ({n/total*100:.1f}%)")

    if transition_steps:
        pct_early = (sum(1 for t in transition_steps if t < N_STEPS * 0.35)
                     / len(transition_steps) * 100)
        print(f"\n   Phase-2 entry: first={min(transition_steps)}  "
              f"median={np.median(transition_steps):.0f}  last={max(transition_steps)}")
        print(f"   Before 35% of training: {pct_early:.0f}%")
        if pct_early >= 50:
            print("   ✅ Phase 2 activates early — curriculum has time to help.")
        else:
            print("   ⚠️  Phase 2 activates late — consider lowering SIGMA_CRITICAL.")
    else:
        print("   ❌ NO Phase-2 transitions — σ never crossed threshold.")

🔍 DIAGNOSIS #3: Phase Transition Timing

ADDITIVE
   P0 Pre-Init     :   102 steps (17.0%)
   P1 Asymmetric   :     0 steps (0.0%)
   P2 Crystallise  :   243 steps (40.5%)
   P3 Intersection :   255 steps (42.5%)

   Phase-2 entry: first=300  median=350  last=400
   Before 35% of training: 100%
   ✅ Phase 2 activates early — curriculum has time to help.

MULTIPLICATIVE
   P0 Pre-Init     :    59 steps (9.8%)
   P1 Asymmetric   :     0 steps (0.0%)
   P2 Crystallise  :   286 steps (47.7%)
   P3 Intersection :   255 steps (42.5%)

   Phase-2 entry: first=150  median=200  last=200
   Before 35% of training: 100%
   ✅ Phase 2 activates early — curriculum has time to help.


In [13]:
# =============================================================================
# CELL 12: Kaggle optimisation summary
# =============================================================================
print("🔧 OPTIMISATION STATUS\n")

checks = [
    ("AMP (Mixed Precision)",     USE_AMP,                  "~1.5-2× speedup on T4"),
    ("cudnn.benchmark",           torch.backends.cudnn.benchmark, "conv kernel autoselect"),
    ("eval_every=50 + fast_eval", True,                     "23× fewer eval passes"),
    ("comp_loader created once",  True,                     "not recreated per run"),
    ("pin_memory DataLoaders",    True,                     "faster host→GPU transfer"),
    ("set_to_none grad zero",     True,                     "slightly less memory"),
    ("enable_nested_tensor=False",True,                     "no UserWarning spam"),
]

for name, status, note in checks:
    icon = "✅" if status else "❌"
    print(f"   {icon} {name:35s} — {note}")

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    resv  = torch.cuda.memory_reserved()  / 1e9
    print(f"\n   GPU memory: {alloc:.2f} GB allocated / {resv:.2f} GB reserved")

print(f"\n📊 Expected runtime estimate:")
print(f"   {N_RUNS_PER_CONDITION * 3} runs × {TIMESTEPS_PER_RUN} steps")
print(f"   With AMP + fast_eval: ~{N_RUNS_PER_CONDITION * 3 * 5:.0f} min total")
print(f"   vs. ~{N_RUNS_PER_CONDITION * 3 * 15:.0f} min without these optimisations")

🔧 OPTIMISATION STATUS

   ✅ AMP (Mixed Precision)               — ~1.5-2× speedup on T4
   ✅ cudnn.benchmark                     — conv kernel autoselect
   ✅ eval_every=50 + fast_eval           — 23× fewer eval passes
   ✅ comp_loader created once            — not recreated per run
   ✅ pin_memory DataLoaders              — faster host→GPU transfer
   ✅ set_to_none grad zero               — slightly less memory
   ✅ enable_nested_tensor=False          — no UserWarning spam

   GPU memory: 0.02 GB allocated / 0.08 GB reserved

📊 Expected runtime estimate:
   45 runs × 2000 steps
   With AMP + fast_eval: ~225 min total
   vs. ~675 min without these optimisations
